In [5]:
# 1: IMPORTS
# =====================================================================
%load_ext autoreload
%autoreload 2

import os 
import pickle
import numpy as np
import pandas as pd
import multiprocessing
from joblib import Parallel, delayed
from scipy.integrate import solve_ivp

from Library.config import BASE_PARAMS
from Library import run_trajectory_grid
from Library.helper_functions import hit_max_blasts

# =====================================================================
# 2: LOAD UNIFIED GAMMAS
# =====================================================================
venex_gamma_df = pd.read_csv("IntermediateResults/venex_gamma_errors.csv")
moffitt_gamma_df = pd.read_csv("IntermediateResults/moffitt_gamma_errors.csv")

v_gamma = venex_gamma_df.loc[venex_gamma_df['global_error'].idxmin(), 'gamma_val']
m_gamma = moffitt_gamma_df.loc[moffitt_gamma_df['global_error'].idxmin(), 'gamma_val']

print(f"Unified VenEx Optimal Gamma: {v_gamma:.2e}")
print(f"Unified Moffitt Optimal Gamma: {m_gamma:.2e}")

# =====================================================================
#  3: PARALLEL GENERATOR & EXECUTION
# =====================================================================
def GetBestGammaTrajectory_Parallel(optimal_gamma, min_time=30):
    # Optimized physiological grid resolution 
    phsc_range = np.arange(0.95, 1.001, 0.01)
    coef_range = np.arange(0.25, 30.05, 0.1)
      
    y0_healthy = [2.42105*10**11, 4*10**8]
    base_params = BASE_PARAMS.copy()

    n_cores = multiprocessing.cpu_count()
    phsc_chunks = np.array_split(phsc_range, n_cores)
    
    results = Parallel(n_jobs=-1)(
        delayed(run_trajectory_grid.run_trajectory_grid)(
            chunk, coef_range, base_params, optimal_gamma, y0_healthy, min_time=min_time
        )
        for chunk in phsc_chunks
    )
    
    return results

if __name__ == '__main__':
    print("Generating Unified VenEx Trajectories...")
    venex_run = GetBestGammaTrajectory_Parallel(v_gamma, min_time=30)
    
    print("Generating Unified Moffitt Trajectories...")
    moffitt_run = GetBestGammaTrajectory_Parallel(m_gamma, min_time=30)

    print("Done! Trajectories generated and ready for saving.")
    
# =====================================================================
#  4: SAVE  TRAJECTORIES
# =====================================================================
    os.makedirs("IntermediateResults", exist_ok=True)
    
    with open("IntermediateResults/venex_best_gamma_runs.pkl", "wb") as f:
        pickle.dump(venex_run, f, protocol=pickle.HIGHEST_PROTOCOL)
        
    with open("IntermediateResults/moffitt_best_gamma_runs.pkl", "wb") as f:
        pickle.dump(moffitt_run, f, protocol=pickle.HIGHEST_PROTOCOL)
        
    print("Successfully saved trajectories to IntermediateResults/")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Unified VenEx Optimal Gamma: 3.16e-06
Unified Moffitt Optimal Gamma: 3.16e-06
Generating Unified VenEx Trajectories...
Generating Unified Moffitt Trajectories...
Done! Trajectories generated and ready for saving.
Successfully saved unified trajectories to IntermediateResults/
